In [ ]:
# parse_bin_as_hex_list.py

def read_binary_as_hex_list(filepath):
    with open(filepath, "rb") as f:
        data = f.read()
        hex_list = [f"0x{byte:02X}" for byte in data]
        print("Parsed Byte Array:")
        print(f"[{', '.join(hex_list)}]")
        return hex_list

if __name__ == "__main__":
    filepath = "crashes/case_7891.bin"  # Update path if needed
    read_binary_as_hex_list(filepath)


In [ ]:
import hashlib
import os


def load_existing_crash_hashes(folder="crashes"):
    crash_hashes = set()
    for fname in os.listdir(folder):
        if fname.endswith(".txt"):  # Only read text-format saved crashes
            path = os.path.join(folder, fname)
            with open(path, "r") as f:
                content = f.read().strip()
                # Normalize spacing, remove brackets, etc. to hash consistently
                normalized = content.replace("[", "").replace("]", "").replace(",", "").replace(" ", "")
                crash_hashes.add(hashlib.md5(normalized.encode()).hexdigest())
    return crash_hashes
crash_hashes = load_existing_crash_hashes()
print(f"Loaded {len(crash_hashes)} unique crash hashes")
print(f"\nc{crash_hashes}")


In [2]:
# STEP 2: Enhanced Fuzzer - Support for unauthenticated + authenticated fuzzing

import logging
import asyncio
import random
import hashlib
import heapq
import os
import sys
from BLEClient import BLEClient
# from ble_mutator import mutate_input
from ble_oracle import is_crash, is_interesting
from UserInterface import ShowUserInterface
from utils import generate_invalid_commands, load_existing_crash_hashes, create_seed_from_command,Seed,inverse_energy
from typing import List, Dict, Set, Tuple, Any, Optional
import time

In [ ]:
# Commands
AUTH = [0x00]
OPEN = [0x01]
CLOSE = [0x02]
DEFAULT_PASSCODE = [0x01, 0x02, 0x03, 0x04, 0x05, 0x06]  # Default passcode

# Define interesting test scenarios
SEED_COMMAND_SEQUENCES:List[List[List[int]]] = [
    # Basic protocol tests
    [AUTH + DEFAULT_PASSCODE],  # Valid authentication
    [OPEN],                     # Open command
    [CLOSE],                    # Close command
    
    # State transition sequences
    [AUTH + DEFAULT_PASSCODE , OPEN],  # Auth + Open
    [AUTH + DEFAULT_PASSCODE , CLOSE], # Auth + Close
    
    # Command sequences in a single frame
    [OPEN , CLOSE],            # Open then Close
    [CLOSE , OPEN],            # Close then Open
    [OPEN , OPEN],             # Open twice
    [CLOSE , CLOSE],           # Close twice
    
    # Complex state transition sequences
    [AUTH + DEFAULT_PASSCODE , OPEN , CLOSE], # Auth + Open + Close
    [AUTH + DEFAULT_PASSCODE , CLOSE , OPEN], # Auth + Close + Open
    
    # Double authentication scenarios (potential bugs)
    [AUTH + DEFAULT_PASSCODE , AUTH + DEFAULT_PASSCODE],
    
    # Full sequences
    [AUTH + DEFAULT_PASSCODE , OPEN , CLOSE , OPEN , CLOSE],
    
    # Edge cases exploration
    [[0x00]],  # AUTH without passcode
    [[0x03]],  # Unknown command (off-by-one from CLOSE)
    [[0xFF]],  # Invalid command (maximum value)
    [AUTH + DEFAULT_PASSCODE[:3]],  # AUTH with incomplete passcode
]

for i, seq in enumerate(SEED_COMMAND_SEQUENCES):

    print(seq)

In [ ]:
def create_seed_from_command(data: List[List[int]], note: str = "") -> Seed:
    """
    Create a Seed from a 2D BLE command sequence.
    - Each command is a list of bytes.
    - The whole sequence is a list of such commands.
    """

    # Flatten for hash: convert [[0x00, 0x01], [0x02]] → bytes([0x00, 0x01, 0x02])
    # flat_bytes = bytes([b for cmd in data for b in cmd])
    # path_hash = hashlib.sha256(flat_bytes).hexdigest()

    return Seed(
        priority=1.0,
        energy=1.0,
        data=data,  # Store 2D structure for proper sequencing
        # path_hash=path_hash,
        mutation_note=note,
        logs=[f"Generated from DEFAULT_COMMAND_SEQUENCES at {time.ctime()}"]
    )

# print("Getting Seed inputs from seed command sequences")
SEED_INPUTS = [
create_seed_from_command(seq, note=f"Predefined test #{i}")
for i, seq in enumerate(SEED_COMMAND_SEQUENCES)
] 
# print(f"Found {len(SEED_INPUTS)} seed inputs.")

for i, seed in enumerate(SEED_INPUTS):
    for i, cmd in enumerate(seed.data):
            hex_cmd = ", ".join(f"0x{b:02X}" for b in cmd)
            print(f"[{hex_cmd}],")
    print("\n")



In [6]:
def is_error(seed: Seed, command: List[int], actual_response: List[int], expected_responses: List[List[int]]) -> bool:
    """
    Determines if the actual BLE response indicates an error by:
    - Checking if it does not match any of the expected valid responses.
    - Checking if authentication using the DEFAULT_PASSCODE fails.
    """

    # 1. General mismatch from expected responses
    if actual_response not in expected_responses:
        return True

    # 2. Explicitly catch failed default authentication
    if command[:1] == AUTH and command[1:] == DEFAULT_PASSCODE:
        if actual_response[0] != 0:
            print("Authentication failed with the correct passcode!")
            # seed.is_error_detected = True
            # seed.error_code = actual_response
            return True

    return False

AUTH = [0x00]  # 7 Bytes
OPEN = [0x01]  # 1 Byte
CLOSE = [0x02]  # 1 Byte
DEFAULT_PASSCODE = [0x01, 0x02, 0x03, 0x04, 0x05, 0x06]  # Correct PASSCODE
# PASSCODE = [0x01, 0x02, 0x03, 0x04, 0x05, 0x07] # Wrong PASSCODE

command = AUTH + DEFAULT_PASSCODE
print(command[:1] == AUTH and command[1:] == DEFAULT_PASSCODE)

actual_response = [0x01]
expected_responses = [[0x00], [0x01], [0x02]]


print(is_error(None, command,actual_response, expected_responses))  # ➜ False


True
Authentication failed with the correct passcode!
True


In [3]:
DEFAULT_PASSCODE = [0x01, 0x02, 0x03, 0x04, 0x05, 0x06]  # Correct PASSCODE

passcode_cmd = bytearray(DEFAULT_PASSCODE)
print(passcode_cmd)


example = bytearray("fsfsfsfsd", "utf-8")
print(example)

bytearray(b'\x01\x02\x03\x04\x05\x06')
bytearray(b'fsfsfsfsd')


In [5]:
example = ['0xc5', '0x1']
print(example)
print(type(example[0]))

['0xc5', '0x1']
<class 'str'>


In [18]:
example = [130]  # 125 is within byte range (0–255)
print(bytearray(example))  # Output: bytearray(b'}')


bytearray(b'\x82')


In [5]:
print("0"*254)

00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
